# SQD Solver with IBM Quantum Backend

This notebook demonstrates how to use the SQD (Sample-based Quantum Diagonalization) solver with IBM Quantum hardware. The purpose of this test is purely to confirm that installation of the software was succesful. The sampling in H2/sto-3g example (2 electrons in 4 spin-orbitals) is trivial and has total of only 6 states in correct particle sector, namely 0011, 0101, 0110, 1001, 1010, 1100. This allows for very quick test of the local deployment of the software.

## Overview

The SQD algorithm combines:
1. **Quantum Sampling**: Build and execute LUCJ ansatz on quantum hardware
2. **Classical Post-Processing**: Diagonalize subspaces using SBD solver

## Prerequisites

Before running this notebook:
1. Create `qpu_config.yaml` from the template
2. Add your IBM Quantum credentials
3. Ensure SBD solver is installed (for post-processing)

## Setup

In [ ]:
import numpy as np
from pyscf import gto, scf, cc, ao2mo

# Import quantum fragment methods
from quantum_fragment_methods.qpu import IBMQuantumBackend
from quantum_fragment_methods.application.solvers.quantum_zoo.sqd import SQDSolver
from quantum_fragment_methods.config import load_qpu_config

## 1. Define Molecular System

We'll use a simple H2 molecule as an example.

In [2]:
# Define H2 molecule
mol = gto.Mole()

mol.atom = '''
H 0 0 0
H 0 0 0.74
'''
mol.basis = 'sto-3g'

mol.build()

print(f"Number of orbitals: {mol.nao}")
print(f"Number of electrons: {mol.nelectron}")

Number of orbitals: 2
Number of electrons: 2


## 2. Run Mean-Field Calculation

Perform Hartree-Fock to get initial guess.

In [3]:
# Hartree-Fock calculation
mf = scf.RHF(mol)
mf.kernel()

print(f"HF energy: {mf.e_tot:.8f}")

converged SCF energy = -1.11675930739643
HF energy: -1.11675931


## 3. Get Hamiltonian Integrals

In [4]:
# Extract Hamiltonian components in MO basis
norb = mol.nao
nelec = (mol.nelec[0], mol.nelec[1])

# The `solve_from_integrals()` method in `sqd.py` (used by the EWF workflow) already expects MO-basis integrals.
mo_coeff = mf.mo_coeff
h1e = mo_coeff.T @ mf.get_hcore() @ mo_coeff
h2e = ao2mo.restore(1, ao2mo.kernel(mol, mo_coeff), norb)

# Get nuclear repulsion energy
nuc_energy = mf.energy_nuc()
# Nuclear repulsion energy needs to be added to SQD results at the end of calculations.
# This change is specific to SQD solver in EWF and deviates from standard Qiskit Addon SQD.

print(f"Number of orbitals: {norb}")
print(f"Total number of electrons: {mol.nelectron}")
print(f"Number of alpha electrons: {nelec[0]}")
print(f"Number of beta electrons: {nelec[1]}")
print(f"One-body tensor shape: {h1e.shape}")
print(f"Two-body tensor shape: {h2e.shape}")
print(f"Nuclear repulsion energy: {nuc_energy:.8f}")

Number of orbitals: 2
Total number of electrons: 2
Number of alpha electrons: 1
Number of beta electrons: 1
One-body tensor shape: (2, 2)
Two-body tensor shape: (2, 2, 2, 2)
Nuclear repulsion energy: 0.71510434


## 4. Load Configuration

Load QPU and SQD configuration from `config.yaml`.

The config file contains:
- **qpu**: IBM Quantum backend settings and credentials
- **sqd**: SQD solver parameters (LUCJ, transpilation, algorithm settings, SBD)
- **ext_sqd**: Extended SQD configuration

In [ ]:
import yaml
from pathlib import Path

# Load configuration from config.yaml in the same directory
# For container workflows, this would be mounted at /workspace/config.yaml
# Create local copy of config_H2_demo.yaml to add you IBM Quantum credentials
config_path: Path = Path('/workspace/quantum-fragment-methods/examples/local_demo/config_local.yaml')

# Load configuration
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Extract configurations
qpu_config = config['qpu']
sqd_config = config['sqd']

print(f"✓ Configuration loaded from: {config_path.absolute()}")
print(f"\nQPU Settings:")
print(f"  Provider: {qpu_config['provider']}")
print(f"  Backend: {qpu_config['backend_name']}")
print(f"  Shots: {qpu_config['sampler_options']['default_shots']:,}")
print(f"\nSQD Settings:")
print(f"  Iterations: {sqd_config['iterations']}")
print(f"  Batches: {sqd_config['n_batches']}")
print(f"  Samples per batch: {sqd_config['samples_per_batch']:,}")

✓ Configuration loaded from: /workspace/quantum-fragment-methods/examples/local_demo/config_local.yaml

QPU Settings:
  Provider: ibm_cleveland
  Backend: ibm_cleveland
  Shots: 50

SQD Settings:
  Iterations: 3
  Batches: 1
  Samples per batch: 20


## 5. Initialize IBM Quantum Backend

Connect to IBM Quantum hardware using the configuration.

In [6]:
# Initialize backend (only run after adding credentials above)
backend = IBMQuantumBackend(qpu_config)
backend.initialize()
backend.get_backend()

# Display backend properties
props = backend.get_backend_properties()
print(f"Backend: {props['backend_name']}")
print(f"Number of qubits: {props['num_qubits']}")
print(f"Basis gates: {props['basis_gates']}")

qiskit_runtime_service._discover_account:WARNING:2026-05-29 16:14:34,946: Loading account with the given token. A saved account will not be used.


Backend: ibm_cleveland
Number of qubits: 156
Basis gates: ['cz', 'id', 'rz', 'sx', 'x']


## 5. Configure SQD Solver

In [7]:
# Extract SQD solver configuration from config.yaml
sqd_config = config['sqd']

print("SQD Configuration:")
print(f"  Iterations: {sqd_config['iterations']}")
print(f"  Batches: {sqd_config['n_batches']}")
print(f"  Samples per batch: {sqd_config['samples_per_batch']}")

# Create SQD solver
solver = SQDSolver(backend, config=sqd_config)
print(f"\nInitialized {solver.name} solver")

SQD Configuration:
  Iterations: 3
  Batches: 1
  Samples per batch: 20

Initialized SQD solver


## 6. Run SQD Calculation

This will:
1. Build LUCJ circuit from CCSD amplitudes
2. Submit to IBM Quantum hardware
3. Retrieve measurement counts
4. Run SBD post-processing

In [ ]:
# Run SQD solver
# Note: This will submit a job to IBM Quantum and may take time to complete

# Use absolute path so results are saved next to this notebook
workflow_dir = '/workspace/quantum-fragment-methods/examples/local_demo/sqd_results'

result = solver.solve(
    h1e=h1e,
    h2e=h2e,
    norb=norb,
    nelec=nelec,
    mf=mf,  # Will compute CCSD amplitudes automatically
    workflow_path=workflow_dir,
    force_resubmit=True # Set to True to force resubmission of the job. 
    # This option must be used when starting the SQD simulation for new molecular system after performing another SQD test.
    # In case of force_resubmit=False the SQD simulation attempts to reuse previously obtained count_dict.npy in "sqd_results" directory.
)

print(f"\nSQD Results:")
print(f"Energy: {result.energy:.8f}")
print(f"Metadata: {result.metadata}")

# Note that for compatibility with EWF framework the nuclear energy is not included in SQD solver, 
# but instead is added at the end of SQD calculations. All energies reported in cell bellow are electronic energies only.

E(CCSD) = -1.13728399861044  E_corr = -0.02052469121401437


/opt/conda/envs/qfrag-env/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py:1001: UserWarning: The backend ibm_cleveland currently has a status of maintenance.
  warnings.warn(


RuntimeError: Failed to submit job: 'Failed to run program: "HTTPSConnectionPool(host=\'quantum.cloud.ibm.com\', port=443): Max retries exceeded with url: /api/v1/jobs (Caused by ResponseError(\'too many 520 error responses\'))"'

## 7. CCSD Comparison

For validation, let's compare the SQD result with classical CCSD.

In [ ]:
# Run CCSD calculation for comparison
mycc = cc.CCSD(mf)
mycc.kernel()

print(f"\nClassical Results:")
print(f"HF energy:   {mf.e_tot:.8f}")
print(f"CCSD energy: {mycc.e_tot:.8f}")
print(f"Correlation energy: {mycc.e_corr:.8f}")

print(f"\nSQD Results")
print(f"SQD energy:  {(result.energy+nuc_energy):.8f}")

# Calculate errors
sqd_error = abs((result.energy+nuc_energy) - mycc.e_tot)
print(f"\nEnergy Comparison:")
print(f"SQD vs CCSD error: {sqd_error:.8f} Ha ({sqd_error * 627.5:.4f} kcal/mol)")

E(CCSD) = -1.13728399861044  E_corr = -0.02052469121401437

Classical Results:
HF energy:   -1.11675931
CCSD energy: -1.13728400
Correlation energy: -0.02052469

SQD Results
SQD energy:  -1.13728383

Energy Comparison:
SQD vs CCSD error: 0.00000016 Ha (0.0001 kcal/mol)

Classical Results:
HF energy:   -1.11675931
CCSD energy: -1.13728400
Correlation energy: -0.02052469

SQD Results
SQD energy:  -1.13728383

Energy Comparison:
SQD vs CCSD error: 0.00000016 Ha (0.0001 kcal/mol)


## Summary

This notebook demonstrated:
- Setting up IBM Quantum backend with configuration
- Creating and configuring SQD solver
- Building LUCJ circuits from CCSD amplitudes
- Submitting jobs to quantum hardware
- Monitoring job status
- Comparing SQD results with classical CCSD for validation